In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sb

In [13]:
df = pd.read_csv("dataset/mnist_train.csv")
print(df.isna().sum().sum())
df

0


,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
X = df.drop("label",axis=1)
y = df["label"]

In [ ]:
class multilayer_perceptron:
    def __init__(self, layer_neurons, learning_rate, epochs, batch_size):

        self.W = []
        self.W.append(np.random.randn(layer_neurons[0],layer_neurons[1]) * np.sqrt(2.0 / layer_neurons[0])) # for ReLU
        self.W.append(np.random.randn(layer_neurons[1],layer_neurons[2]) * np.sqrt(1.0 / layer_neurons[1])) # for tanh
        self.W.append(np.random.randn(layer_neurons[2],layer_neurons[3]) * np.sqrt(1.0 / layer_neurons[2])) # for softmax
        
        self.b = []
        for i in range(3):
            self.b.append(np.zeros((1,layer_neurons[i+1])))

        self.A = []
        self.Z = []

        self.layers_n = layer_neurons
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        self.results = []
    
    # Activation Functions

    def ReLU(self, X):
        return np.maximum(0,X)
    
    def ReLU_derivative(self, A):
        return (A > 0) * 1.0


    def tanh(self, X):
        a = np.exp(X)
        b = np.exp(-X)
        return (a - b) / (a + b)

    def tanh_derivative(self, A):
        return 1.0 - A**2 # = sech^2(X) = tanh_derivative(X)


    def softmax(self, X):
        row_maxes = np.max(X,axis=1,keepdims=True)
        shifted_X = X - row_maxes
        exp_X = np.exp(shifted_X)
        return exp_X / np.sum(exp_X,axis=1,keepdims=True)

    # -------

    def feed_forward(self, X):
        self.A = [X]  
        self.Z = []  # preactivation
        
        # Hidden Layer 1 (ReLU)
        z1 = np.dot(self.A[0], self.W[0]) + self.b[0]
        a1 = self.ReLU(z1)
        self.Z.append(z1)
        self.A.append(a1)
        
        # Hidden Layer 2 (Tanh)
        z2 = np.dot(self.A[1], self.W[1]) + self.b[1]
        a2 = self.tanh(z2)
        self.Z.append(z2)
        self.A.append(a2)
        
        # Output Layer (Softmax)
        z3 = np.dot(self.A[2], self.W[2]) + self.b[2]
        a3 = self.softmax(z3)
        self.Z.append(z3)
        self.A.append(a3)
        
        return a3

    def _backpropagate(self, Y):

        total_samples = Y.shape[0]

        dZ3 = self.A[3] - Y
        dW3 = np.dot(self.A[2].T,dZ3) / total_samples
        db3 = np.sum(dZ3,axis=0,keepdims=True) / total_samples

        dZ2 = np.dot(dZ3,self.W[2].T) * self.tanh_derivative(self.A[2])
        dW2 = np.dot(self.A[1].T,dZ2) / total_samples
        db2 = np.sum(dZ2,axis=0,keepdims=True) / total_samples

        dZ1 = np.dot(dZ2,self.W[1].T) * self.ReLU_derivative(self.Z[0])
        dW1 = np.dot(self.A[0].T,dZ1) / total_samples
        db1 = np.sum(dZ1,axis=0,keepdims=True) / total_samples

        self.W[2] -= self.lr * dW3
        self.b[2] -= self.lr * db3

        self.W[1] -= self.lr * dW2
        self.b[1] -= self.lr * db2

        self.W[0] -= self.lr * dW1
        self.b[0] -= self.lr * db1
    
    def compute_loss(self, Y_pred, Y_true):
        epsilon = 1e-15
        Y_pred = np.clip(Y_pred, epsilon, 1 - epsilon)
        
        loss = -np.sum(Y_true * np.log(Y_pred)) / Y_true.shape[0]
        return loss

    def fit(self,X,Y):
        
        total_samples = Y.shape[0]
        indices = np.arange(total_samples)

        for ep in range(self.epochs):
            np.random.shuffle(indices)
            X_shuffled = X[indices]
            Y_shuffled = Y[indices]

            for i in range(0, total_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                Y_batch = Y_shuffled[i : i + self.batch_size]

                self.feed_forward(X_batch)
                self._backpropagate(Y_batch)            

            full_predictions = self.feed_forward(X)
            epoch_loss = self.compute_loss(full_predictions, Y)
            self.results.append(epoch_loss)

        










        


IndentationError: expected an indented block after function definition on line 36 (1540673245.py, line 39)